In [11]:
# При обращении к make_averager  возвращается объект-функция averager. При каждом вызове averager добавляет переданный аргумент в конец списка series 
# и вычисляет текущее среднее
def make_averager():
    series = []
    def averager(new_value):
        series.append(new_value)
        total = sum(series)
        return total / len(series)
    return averager

In [12]:
avg=make_averager()

In [14]:
avg(100)

60.5

In [15]:
avg.__code__.co_freevars

('series',)

In [16]:
avg.__closure__

(<cell at 0x0000020D07E42260: list object at 0x0000020D0848AE40>,)

In [17]:
avg.__closure__[0].cell_contents

[21, 100]

In [3]:
# Декоратор с без параметров, принимает любую функцию

#  Синтаксис @deco автоматически передаёт функцию foo первым аргументом в deco. 
# Обёртка wrapper просто перехватывает вызов, добавляет логику и отдаёт управление оригинальной функции.

import time

def deco(func):
    def wrapper(*args, **kwargs):
        start=time.time()
        time.sleep(3)
        result=func(*args, **kwargs)
        end=time.time()
        print(f'Время выполнения: {end-start}')
        return result
    return wrapper

@deco
def foo():
    return 42 


foo()

Время выполнения: 3.0004220008850098


42

In [5]:
# Декоратор с параметрами

# Python сначала вызывает delay(2). Эта функция должна вернуть другой декоратор (нашу функцию decorator), 
# который уже примет foo. Поэтому структура всегда: параметры → функция → аргументы функции.

import time

def delay(seconds):
    # Уровень 1: Принимает параметры самого декоратора
    def decorator(func):
        # Уровень 2: Принимает саму функцию
        def wrapper(*args, **kwargs):
            # Уровень 3: Принимает аргументы вызова функции
            print(f"Ждём {seconds} секунд...")
            time.sleep(seconds)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@delay(seconds=2)
def foo():
    return 42

foo()


Ждём 2 секунд...


42

In [ ]:
# Декоратор на основе класса (с сохранением состояния)

# __init__ принимает декорируемую функцию (как Уровень 1). 
# __call__ делает класс "вызываемым" объектом, принимая аргументы функции (как Уровень 2). 
# Главный плюс: легко хранить переменные состояния (self.count) без использования сложных конструкций вроде nonlocal.


import time

class CallCounter:
    def __init__(self, func):
        # Сохраняем функцию и инициализируем состояние
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        # Этот метод срабатывает при вызове функции
        self.count += 1
        print(f"Функция {self.func.__name__} вызвана {self.count} раз(а)")
        
        start = time.time()
        result = self.func(*args, **kwargs)
        print(f"Время: {time.time() - start:.2f} сек.")
        return result

@CallCounter
def foo():
    time.sleep(0.5)
    return 42

foo()
foo()


Функция foo вызвана 1 раз(а)
Время: 0.50 сек.
Функция foo вызвана 2 раз(а)
Время: 0.50 сек.


42

In [7]:
# Декоратор, изменяющий данные

# Декоратор выступает как фильтр. Он позволяет выполнить исходную функцию, 
# перехватить её result, изменить его (или аргументы args до вызова) и вернуть уже модифицированное значение.


def to_uppercase(func):
    def wrapper(*args, **kwargs):
        # 1. Перехватываем результат
        result = func(*args, **kwargs)
        
        # 2. Модифицируем его перед возвратом
        if isinstance(result, str):
            return result.upper()
        return result
    return wrapper

@to_uppercase
def get_greeting(name):
    return f"привет, {name}"

get_greeting("мир")


'ПРИВЕТ, МИР'